In [1]:
# 2조 API 키 - 제일 먼저 실행해주기
import os 
os.environ["OPENAI_API_KEY"] = ""

# 모델 경로 챗봇 모델

In [8]:
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain_chroma import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import json


def start_chatbot(db_path):
    # 1. 벡터스토어 로드
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")
    vectordb = Chroma(persist_directory=db_path, embedding_function=embedding)

    # 2. LLM 준비
    llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

    # 3. MultiQueryRetriever 적용
    retriever = MultiQueryRetriever.from_llm(
        retriever=vectordb.as_retriever(search_kwargs={"k": 3}),
        llm=llm
    )

    # 4. QA 체인 구성
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        return_source_documents=True
    )


    # 국가 리스트
    COUNTRY_LIST = [
        "베트남", "싱가포르", "말레이시아", "인도네시아", "태국",
        "필리핀", "캄보디아", "라오스", "브루나이", "미얀마",
        "그리스", "네덜란드", "덴마크", "독일", "라트비아",
        "루마니아", "룩셈부르크", "리투아니아", "몰타", "벨기에",
        "불가리아", "사이프러스", "스웨덴", "스페인", "슬로바키아",
        "슬로베니아", "아일랜드", "에스토니아", "오스트리아", "이탈리아",
        "체코공화국", "크로아티아", "포르투갈", "폴란드", "프랑스",
        "핀란드", "헝가리"
    ]

    # 쿼리에서 국가명 추출
    def extract_country(query):
        for country in COUNTRY_LIST:
            if country in query:
                return country
        return None

    # 키워드 기반 청크 검색
    def search_chunks_by_keyword(vectordb, keyword, limit=1000):
        # print(f"🔍 키워드 기반 청크 검색: '{keyword}'")
        results = []
        all_docs = vectordb.get(include=["documents", "metadatas"], limit=limit)
        for i, text in enumerate(all_docs["documents"]):
            if keyword in text:
                # dict 형태여서 document 객체로 리턴해줘야미
                results.append(Document(page_content=text, metadata=all_docs["metadatas"][i])) 
        return results

    # 키워
    def answer_from_context(query, context_chunks):
        context = "\n\n".join(doc.page_content for doc in context_chunks[:5])  # 상위 5개
        prompt = f"""
            아래는 통계 설명입니다. 참고해서 사용자 질문에 답변해주세요.
            
            설명:
            {context}
            
            질문:
            {query}
            """
        return llm.invoke(prompt).content

    # 챗봇
    print(" 챗봇 준비 완료! 종료하려면 '종료' 입력.")
    last_doc = None

    while True:
        query = input(" 질문: ")
        if query.lower() in ["종료", "exit"]:
            print("👋 챗봇을 종료합니다.")
            break

        # 1️⃣ 키워드 기반 선 필터링
        country = extract_country(query)
        
        if country:
            keyword_docs = search_chunks_by_keyword(vectordb, country)
            if keyword_docs:
                # print(f"✅ '{country}' 관련 문서 {len(keyword_docs)}건 발견 → 직접 주입")
                response = answer_from_context(query, keyword_docs)
                # print(f"\n⏱️ 응답 시간: {time.time() - start_time:.2f}초")
                print("", response)
                continue
                
        
        # 후속 질문: 표/그래프/지도
        if last_doc:
            if "표" in query and "설명" in query:
                print("\n📊 [표 설명]\n" + (last_doc.metadata.get("table_info") or "❌ 설명 없음"))
                continue
            if "그래프" in query and "설명" in query:
                print("\n📈 [그래프 설명]\n" + (last_doc.metadata.get("graph_info") or "❌ 설명 없음"))
                continue
            if "지도" in query and "설명" in query:
                print("\n🗺️ [지도 설명]\n" + (last_doc.metadata.get("map_info") or "❌ 설명 없음"))
                continue

        # 응답 시간 측정
        start_time = time.time()
        
        # 응답 생성
        result = qa_chain.invoke({"query": query})

        # 응답 시간 측정 종료 
        elapsed = time.time() - start_time
        print(f"\n⏱️ 응답 시간: {elapsed:.2f}초")

        
        # # 검색 청크 출력
        # if result.get("source_documents"):
        #     print("\n📄 [검색된 청크 내용 미리보기]")
        #     print(result["source_documents"])
        #     for doc in result["source_documents"]:
        #         print("🟨", doc.metadata.get("menu_path", "알 수 없음"))
        #         print(doc.page_content[:300], "...")

        print("\n🧠 답변:\n", result["result"])
        print("\n\n")

        # 저장
        if result.get("source_documents"):
            last_doc = result["source_documents"][0]   # 가장 유사한 청크 저장 
            meta = last_doc.metadata
            if meta.get("login_required") == "일반회원":
                print("⚠️ 일반회원 로그인 필요")
            elif meta.get("login_required") == "유료회원":
                print("🔒 유료회원 전용")
            if meta.get("url"):
                print(f"🔗 링크: {meta['url']}")
            if any([meta.get("table_info"), meta.get("graph_info"), meta.get("map_info")]):
                print("📎 표, 그래프, 지도 설명이 필요하시면 말씀해주세요!")

# 챗봇 실행!

In [ ]:
if __name__ == "__main__":
    start_chatbot("./datasets/chroma_db_1500_300")

 챗봇 준비 완료! 종료하려면 '종료' 입력.


 질문:  미국의 수출 통계가 궁금해



⏱️ 응답 시간: 13.44초

🧠 답변:
 미국의 수출 통계를 확인하고 싶으시다면, 'IMF 세계통계 → 무역통계로 보는 주요국 → 미국 → 미국의 10대 수출상품' 메뉴를 통해 미국의 연도별 주요 수출품목을 품목별 수출액 및 비중 기준으로 확인할 수 있습니다. 이 메뉴는 1988년부터의 데이터를 포함하며, 막대형, 꺾은선형, 도넛형 그래프를 지원하여 미국의 산업 구조 및 글로벌 수출 경쟁력을 품목별로 분석하는 데 유용합니다. 해당 데이터는 다음 링크에서 확인하실 수 있습니다:

🔗 [미국의 10대 수출상품](https://stat.kita.net/stat/world/major/USStats06.screen)
🔗 링크: https://stat.kita.net/stat/world/major/USStats06.screen
📎 표, 그래프, 지도 설명이 필요하시면 말씀해주세요!


 질문:  표에 대해서 설명해줘



📊 [표 설명]
표 구성은 다음과 같다.
- 년도(순위): 기준 연도 및 품목 순위
- 상품: 수출 품목명 (HS 4단위 기준)
- 10대 상품 수출액(백만불): 상위 10개 품목의 수출 총액
- 비중(%): 해당 품목이 총수출에서 차지하는 비율
- 총 수출액(백만불): 미국 전체 수출 규모 (하단 참고)


 질문:  말레이시아의 수입 통계가 궁금해


 말레이시아와의 수입 통계를 확인하고 싶으시다면, KITA의 해외무역통계 서비스를 이용하실 수 있습니다. 아래의 절차를 따라 말레이시아의 수입 통계를 조회할 수 있습니다:

1. 해외무역통계 → 아시아 → ASEAN → 국가별 메뉴로 이동하세요.
2. 말레이시아를 선택하여 해당 국가와의 연도별 수입 실적을 확인할 수 있습니다.
3. 수출입 금액, 증감률, 무역수지 등의 정보를 국가 단위로 비교할 수 있으며, 기본 정렬 기준은 수출금액 내림차순입니다.
4. 추가적으로 품목, 단위, 정렬기준 등을 설정하여 더 상세한 데이터를 조회할 수 있습니다.

해당 서비스는 KITA 기업회원(유료회원사) 전용이므로, 서비스 이용을 위해서는 회원 가입이 필요합니다. 자세한 정보는 다음 링크를 통해 확인하실 수 있습니다:
🔗 https://stat.kita.net/stat/istat/asean/AseanCtrImpExpList.screen

또한, 일부 국가의 무역통계 상세자료 발표가 지연되고 있어 최신 데이터 제공에 어려움이 있을 수 있음을 양해 부탁드립니다.


# 1 
챗봇 준비 완료! 종료하려면 '종료' 입력.
 질문:  미국의 수출 통계가 궁금해

⏱️ 응답 시간: 13.44초

🧠 답변:
 미국의 수출 통계를 확인하고 싶으시다면, 'IMF 세계통계 → 무역통계로 보는 주요국 → 미국 → 미국의 10대 수출상품' 메뉴를 통해 미국의 연도별 주요 수출품목을 품목별 수출액 및 비중 기준으로 확인할 수 있습니다. 이 메뉴는 1988년부터의 데이터를 포함하며, 막대형, 꺾은선형, 도넛형 그래프를 지원하여 미국의 산업 구조 및 글로벌 수출 경쟁력을 품목별로 분석하는 데 유용합니다. 해당 데이터는 다음 링크에서 확인하실 수 있습니다:

🔗 [미국의 10대 수출상품](https://stat.kita.net/stat/world/major/USStats06.screen)
🔗 링크: https://stat.kita.net/stat/world/major/USStats06.screen
📎 표, 그래프, 지도 설명이 필요하시면 말씀해주세요!

 질문:  표에 대해서 설명해줘

📊 [표 설명]
표 구성은 다음과 같다.
- 년도(순위): 기준 연도 및 품목 순위
- 상품: 수출 품목명 (HS 4단위 기준)
- 10대 상품 수출액(백만불): 상위 10개 품목의 수출 총액
- 비중(%): 해당 품목이 총수출에서 차지하는 비율
- 총 수출액(백만불): 미국 전체 수출 규모 (하단 참고)

질문: 말레이시아의 수입 통계가 궁금해
 말레이시아와의 수입 통계를 확인하고 싶으시다면, KITA의 해외무역통계 서비스를 이용하실 수 있습니다. 아래의 절차를 따라 말레이시아의 수입 통계를 조회할 수 있습니다:

1. 해외무역통계 → 아시아 → ASEAN → 국가별 메뉴로 이동하세요.
2. 말레이시아를 선택하여 해당 국가와의 연도별 수입 실적을 확인할 수 있습니다.
3. 수출입 금액, 증감률, 무역수지 등의 정보를 국가 단위로 비교할 수 있으며, 기본 정렬 기준은 수출금액 내림차순입니다.
4. 추가적으로 품목, 단위, 정렬기준 등을 설정하여 더 상세한 데이터를 조회할 수 있습니다.

해당 서비스는 KITA 기업회원(유료회원사) 전용이므로, 서비스 이용을 위해서는 회원 가입이 필요합니다. 자세한 정보는 다음 링크를 통해 확인하실 수 있습니다:
🔗 https://stat.kita.net/stat/istat/asean/AseanCtrImpExpList.screen

또한, 일부 국가의 무역통계 상세자료 발표가 지연되고 있어 최신 데이터 제공에 어려움이 있을 수 있음을 양해 부탁드립니다.